In [1]:
import os
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, AutoConfig, AutoTokenizer, AutoModelForCausalLM,AutoModelForVision2Seq
from qwen_vl_utils import process_vision_info
import pandas as pd
import json
import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# !pip install -U "huggingface_hub[hf_transfer]" -q
# !pip install "huggingface_hub[cli]"
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # high speed download
# !pip install -U huggingface_hub -q
# !huggingface-cli login

DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print(DTYPE)

torch.bfloat16


In [2]:
def load_qwen_model(model_id: str, device: str | None = None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading Qwen model: {model_id}")

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=DTYPE,
        device_map="auto"
    )
    model.eval()

    print("Qwen model loaded successfully!")
    return model, tokenizer, processor


def load_internvl_model(model_id: str = "OpenGVLab/InternVL3_5-8B-HF"):
    print(f"Loading InternVL HF model: {model_id}")
    from transformers import AutoProcessor, AutoModelForImageTextToText
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        device_map="auto",  # None on CPU
    )
    print("InternVL HF model loaded successfully!")
    return model, processor




def qwen_ask(image_path: str, prompt: str, *, model, processor, max_new_tokens: int = 64) -> str:
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=int(max_new_tokens))

    gen = outputs[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(gen, skip_special_tokens=True).strip()


@torch.inference_mode()
def internvl_ask(image_path: str, prompt: str, *, model, processor, max_new_tokens: int = 128):
    # print("Processing image and prompt...")
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    # print("Applying chat template...")
    inputs_raw = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # print("Moving inputs to device...")
    inputs = {}
    for k, v in inputs_raw.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(model.device)
        else:
            inputs[k] = v

    # print(f"Starting generation (max_new_tokens={max_new_tokens})...")
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    # print("Generation completed!")

    input_len = inputs["input_ids"].shape[-1]
    answer = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
    return answer.strip()



def show_image(image_path: str, title: str = ""):
    img = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

### read json (valid image paths)

In [3]:
# read valide_image_paths.json
with open("valide_image_paths.json", "r") as f:
    valide_image_paths = json.load(f)



#### Test Qwen2.5-vl-7B

In [4]:
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
model_qwen, tokenizer_qwen, processor_qwen = load_qwen_model(QWEN_MODEL_ID)

Loading Qwen model: Qwen/Qwen2.5-VL-7B-Instruct


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
/home/zzou/.dataset/lib/python3.12/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Qwen model loaded successfully!


In [5]:
import os
import re
import json
import csv
from typing import Dict, Tuple

# ---------- helpers ----------

def yn(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.split(r"\s+", x)[0].strip(".,;:!()[]{}<>\"'`")
    if x in ("yes", "y", "true", "1"):
        return "yes"
    if x in ("no", "n", "false", "0"):
        return "no"
    return x  # fallback

def parse_relation_from_basename(base: str) -> Tuple[str, str, str]:
    """
    Parse patterns like:
      book_right_of_chair_FACE-CAMERA
      beer-bottle_left_of_chair_FACE-LEFT
    """
    # split at '_FACE-' (your examples)
    core = base.split("_FACE-", 1)[0]
    second_object = core.split("_")[0] 
    direction1 = core.split("_")[1] 
    direction2 = base.split("_FACE-", 1)[1]

    if direction2 == "CAMERA":
        if direction1 == "right":
            correct_relation = "left"
        elif direction1 == "left":
            correct_relation = "right"
    elif direction2 == "LEFT":
        if direction1 == "right":
            correct_relation = "behind"
        elif direction1 == "left":
            correct_relation = "front"
    elif direction2 == "RIGHT":
        if direction1 == "right":
            correct_relation = "front"
        elif direction1 == "left":
            correct_relation = "behind"

    return second_object, correct_relation

# ---------- run loop + save CSV ----------

output_csv = "spatial_answers_qwen2.5-7B.csv"

with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    header = [
        "image_path",
        "second_object",
        "where_answer",

        "front_answer",
        "back_answer",
        "left_answer",
        "right_answer",

        "correct_answer",
    ]
    writer.writerow(header)

    for img_path in valide_image_paths:
        base = os.path.splitext(os.path.basename(img_path))[0]
        second_object, correct_relation = parse_relation_from_basename(base)

        # prompts
        where_prompt = f"Where is the {second_object} in the view of the human in the image?"
        front_prompt = f"Is the {second_object} in the front of the human? Answer only yes or no."
        back_prompt  = f"Is the {second_object} in the back of the human? Answer only yes or no."
        left_prompt  = f"Is the {second_object} on the left of the human? Answer only yes or no."
        right_prompt = f"Is the {second_object} on the right of the human? Answer only yes or no."

        # ask model
        where_ans = qwen_ask(img_path, where_prompt, model=model_qwen, processor=processor_qwen, max_new_tokens=64)
        front_ans = yn(qwen_ask(img_path, front_prompt, model=model_qwen, processor=processor_qwen, max_new_tokens=8))
        back_ans  = yn(qwen_ask(img_path, back_prompt,  model=model_qwen, processor=processor_qwen, max_new_tokens=8))
        left_ans  = yn(qwen_ask(img_path, left_prompt,  model=model_qwen, processor=processor_qwen, max_new_tokens=8))
        right_ans = yn(qwen_ask(img_path, right_prompt, model=model_qwen, processor=processor_qwen, max_new_tokens=8))

        row = [
            img_path,
            second_object,
            where_ans,

            front_ans,
            back_ans,
            left_ans,
            right_ans,

            correct_relation,
        ]
        writer.writerow(row)

print(f"Saved: {output_csv}")

Saved: spatial_answers_qwen2.5-7B.csv
